# Diviser pour régner — Exercice 2 : le tri fusion

**Notions travaillées :** parcours de tableaux avec indices, fusion de deux tableaux triés, récursivité, version itérative, coût en $n\log(n)$.

> **Principe du tri fusion (*merge sort*)**
> - **Diviser** : découper le tableau en deux moitiés ;
> - **Régner** : trier chaque moitié (récursivement ; un tableau de taille 0 ou 1 est déjà trié) ;
> - **Combiner** : fusionner les deux moitiés triées.

## Question 1 : `est_trie(t)`

Un tableau est trié dans l'ordre croissant si **chaque élément est inférieur ou égal au suivant**. On compare donc `t[i]` et `t[i+1]` pour `i` allant de 0 à `len(t) - 2`.

💡 C'est un algorithme « **pour tout** » : on renvoie `False` **dès le premier** couple mal rangé, et `True` seulement après avoir tout vérifié.

⚠️ **Erreurs fréquentes :**
- `range(len(t))` : à la dernière valeur de `i`, `t[i+1]` sort du tableau (`IndexError`) ;
- mettre `return True` dans un `else` à l'intérieur de la boucle : la fonction s'arrête dès la première comparaison.

In [1]:
def est_trie(t):
    for i in range(len(t) - 1):
        if t[i] > t[i + 1]:
            return False
    return True


assert est_trie([1, 2, 2, 5])
assert not est_trie([1, 3, 2])
assert est_trie([]) and est_trie([7])     # cas limites : vide et un seul élément

## Question 2 : `partie_gauche` et `partie_droite` sans « pythonnade »

On n'a pas le droit au *slicing* (`t[:n//2]`). On construit donc le résultat **élément par élément** avec une boucle et `append`.

- partie gauche : indices de `0` à `n//2 - 1` → `range(n // 2)` ;
- partie droite : indices de `n//2` à `n - 1` → `range(n // 2, n)`, soit `n - n//2` éléments.

💡 Pour une taille impaire, par exemple $n = 7$ : gauche = 3 éléments, droite = 4 éléments. Les deux parties réunies redonnent bien tout le tableau.

In [2]:
def partie_gauche(t):
    n = len(t)
    res = []
    for i in range(n // 2):
        res.append(t[i])
    return res


def partie_droite(t):
    n = len(t)
    res = []
    for i in range(n // 2, n):
        res.append(t[i])
    return res


t = [8, 3, 2, 1, 5, 4, 7]
print(partie_gauche(t), partie_droite(t))
assert partie_gauche(t) + partie_droite(t) == t

[8, 3, 2] [1, 5, 4, 7]


## Question 3 : la fonction `fusion(t1, t2)`

**Idée :** on utilise deux « doigts », un indice `i1` dans `t1` et un indice `i2` dans `t2`.
- Tant que les deux tableaux ont encore des éléments, on compare `t1[i1]` et `t2[i2]`, on ajoute **le plus petit** au résultat et on avance le doigt correspondant.
- Quand l'un des tableaux est épuisé, on recopie **la fin de l'autre** (elle est déjà triée).

Exemple avec `[2, 5, 7, 9]` et `[3, 8, 10]` : 2 < 3 → 2 ; 5 > 3 → 3 ; 5 < 8 → 5 ; 7 < 8 → 7 ; 9 > 8 → 8 ; 9 < 10 → 9 ; `t1` est épuisé → on recopie 10.

✅ **Respect des contraintes :**
- `t1` et `t2` ne sont **pas modifiés** : on lit seulement leurs éléments, on n'utilise jamais `pop` ni `remove` ;
- chaque comparaison ajoute un élément au résultat, donc il y a **au plus** `len(t1) + len(t2)` comparaisons, de l'ordre de `max(len(t1), len(t2))` ;
- aucun algorithme de tri n'est utilisé.

⚠️ L'énoncé contient une coquille : la dernière ligne appelle `est_triee(t)` alors que la fonction s'appelle `est_trie`. Corrigée ici.

In [3]:
def fusion(t1, t2):
    assert est_trie(t1), "le tableau t1 n'est pas trié"
    assert est_trie(t2), "le tableau t2 n'est pas trié"
    t = []
    i1, i2 = 0, 0
    while i1 < len(t1) and i2 < len(t2):
        if t1[i1] <= t2[i2]:
            t.append(t1[i1])
            i1 += 1
        else:
            t.append(t2[i2])
            i2 += 1
    # un des deux tableaux est épuisé : on recopie la fin de l'autre
    while i1 < len(t1):
        t.append(t1[i1])
        i1 += 1
    while i2 < len(t2):
        t.append(t2[i2])
        i2 += 1
    assert est_trie(t), "le tableau t n'est pas trié"
    return t


a, b = [2, 5, 7, 9], [3, 8, 10]
print(fusion(a, b))
assert fusion(a, b) == [2, 3, 5, 7, 8, 9, 10]
assert a == [2, 5, 7, 9] and b == [3, 8, 10]     # t1 et t2 non modifiés
assert fusion([], [1, 2]) == [1, 2]

[2, 3, 5, 7, 8, 9, 10]


## Question 4

### (a) Sur papier : tri fusion de `[8, 3, 2, 1, 5, 4, 7, 6]`

```
Découpage (diviser)                 Fusions (combiner)
[8, 3, 2, 1, 5, 4, 7, 6]            [1, 2, 3, 4, 5, 6, 7, 8]
[8, 3, 2, 1] [5, 4, 7, 6]           [1, 2, 3, 8] [4, 5, 6, 7]
[8, 3] [2, 1] [5, 4] [7, 6]         [3, 8] [1, 2] [4, 5] [6, 7]
[8] [3] [2] [1] [5] [4] [7] [6]     (tableaux de taille 1 : déjà triés)
```

On descend en coupant en deux jusqu'à des tableaux de taille 1, puis on **remonte en fusionnant** deux à deux. La fonction ci-dessous affiche cette trace automatiquement.

### (b) `tri_fusion_rec(t)`

La fonction récursive suit exactement le principe :
- **cas de base** : si `len(t) <= 1`, le tableau est déjà trié, on renvoie une **copie** (on doit renvoyer un **nouveau** tableau) ;
- **sinon** : on coupe, on trie chaque moitié **par un appel récursif**, et on fusionne.

💡 **Pourquoi ça termine ?** À chaque appel, la taille est divisée par 2 : on finit forcément par atteindre une taille ⩽ 1.

In [4]:
def tri_fusion_rec(t, profondeur=0, trace=False):
    if trace:
        print("    " * profondeur + "trie", t)
    if len(t) <= 1:
        return [x for x in t]          # nouveau tableau (copie)
    gauche = tri_fusion_rec(partie_gauche(t), profondeur + 1, trace)
    droite = tri_fusion_rec(partie_droite(t), profondeur + 1, trace)
    res = fusion(gauche, droite)
    if trace:
        print("    " * profondeur + "fusionne", gauche, "+", droite, "->", res)
    return res


t = [8, 3, 2, 1, 5, 4, 7, 6]
print(tri_fusion_rec(t, trace=True))
print("t n'a pas été modifié :", t)

trie [8, 3, 2, 1, 5, 4, 7, 6]
    trie [8, 3, 2, 1]
        trie [8, 3]
            trie [8]
            trie [3]
        fusionne [8] + [3] -> [3, 8]
        trie [2, 1]
            trie [2]
            trie [1]
        fusionne [2] + [1] -> [1, 2]
    fusionne [3, 8] + [1, 2] -> [1, 2, 3, 8]
    trie [5, 4, 7, 6]
        trie [5, 4]
            trie [5]
            trie [4]
        fusionne [5] + [4] -> [4, 5]
        trie [7, 6]
            trie [7]
            trie [6]
        fusionne [7] + [6] -> [6, 7]
    fusionne [4, 5] + [6, 7] -> [4, 5, 6, 7]
fusionne [1, 2, 3, 8] + [4, 5, 6, 7] -> [1, 2, 3, 4, 5, 6, 7, 8]
[1, 2, 3, 4, 5, 6, 7, 8]
t n'a pas été modifié : [8, 3, 2, 1, 5, 4, 7, 6]


In [5]:
# Test sur des tableaux aléatoires : on compare avec sorted
from random import randint
for _ in range(200):
    t = [randint(0, 50) for _ in range(randint(0, 30))]
    assert tri_fusion_rec(t) == sorted(t)
print("200 tests réussis ✔")

200 tests réussis ✔


## Question 5 : version itérative `tri_fusion_it(t)`

On fait le travail **de bas en haut**, sans récursivité :
1. on découpe le tableau en tableaux de taille 1 (déjà triés) ;
2. on **fusionne les tableaux deux par deux** : on obtient des tableaux triés de taille 2, puis 4, puis 8... ;
3. on s'arrête quand il ne reste **qu'un seul** tableau.

S'il reste un tableau seul (nombre impair de morceaux), il passe tel quel à l'étape suivante.

```
[8] [3] [2] [1] [5] [4] [7] [6]
[3, 8] [1, 2] [4, 5] [6, 7]
[1, 2, 3, 8] [4, 5, 6, 7]
[1, 2, 3, 4, 5, 6, 7, 8]
```

In [6]:
def tri_fusion_it(t):
    if len(t) == 0:
        return []
    morceaux = [[x] for x in t]            # étape 1 : n tableaux de taille 1
    while len(morceaux) > 1:
        suivants = []
        for i in range(0, len(morceaux) - 1, 2):
            suivants.append(fusion(morceaux[i], morceaux[i + 1]))
        if len(morceaux) % 2 == 1:          # morceau seul en fin de liste
            suivants.append(morceaux[-1])
        morceaux = suivants
        print(morceaux)
    return morceaux[0]


print(tri_fusion_it([8, 3, 2, 1, 5, 4, 7, 6]))

[[3, 8], [1, 2], [4, 5], [6, 7]]
[[1, 2, 3, 8], [4, 5, 6, 7]]
[[1, 2, 3, 4, 5, 6, 7, 8]]
[1, 2, 3, 4, 5, 6, 7, 8]


In [7]:
import io
from contextlib import redirect_stdout
with redirect_stdout(io.StringIO()):          # on coupe l'affichage pendant les tests
    for _ in range(200):
        t = [randint(0, 50) for _ in range(randint(0, 30))]
        assert tri_fusion_it(t) == sorted(t)
print("200 tests réussis ✔")

200 tests réussis ✔


## Pourquoi le tri fusion est-il efficace ? (lien avec la Proposition du cours)

Pour $n = 2^k$ :
- il y a $k = \log(n)$ **niveaux** de découpage ;
- à chaque niveau, les fusions traitent en tout les $n$ éléments, soit environ $n$ comparaisons.

Coût total : $n \times \log(n)$, à comparer avec $n^2$ pour un tri par sélection ou par insertion. Pour $n = 10^6$ : environ $2 \times 10^7$ opérations au lieu de $10^{12}$.

**Proposition :** le coût moyen du tri fusion d'un tableau de taille $n$ est en $n\log(n)$.